# 01 — Base/Gold behavior smoke and manual gate

**Goal.** Load the frozen `Qwen/Qwen3.6-27B` base once, attach only the
published Gold Taboo LoRA, and establish the first organism before reading
activations. Blue is deliberately deferred until after base J-Lens sanity.

This notebook uses only prompts copied from the four published Taboo splits.
It runs three base/Gold smoke prompts for human inspection and requires an
explicit saved approval before notebook 02 can load the lens.

**What this notebook does not establish:** it does not show that a secret is
decodable internally. It only validates the organism and records leakage.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


## Create an immutable run

Every execution gets a new timestamped run directory. Copy the printed
`RUN_ID` into notebooks 02–04. Re-running cells within this run is resumable;
starting this cell again intentionally creates a new run rather than
overwriting an older one.


In [ ]:
from src.experiment_io import create_run
from src.preflight import runtime_dependency_preflight, static_preflight

paths = create_run("configs/gold_blue_experiment.json")
RUN_ID = paths.run_id
print("RUN_ID =", RUN_ID)
print("results:", paths.result_dir)


In [ ]:
preflight = static_preflight("configs/gold_blue_experiment.json")
display(preflight)
assert preflight["passed"], "Static preflight failed; inspect the report before loading weights."
(paths.result_dir / "gold_blue_static_preflight.json").write_text(
    json.dumps(preflight, indent=2), encoding="utf-8"
)

import shutil
for relative in (
    "results/artifact_preflight.json",
    "results/environment_report.json",
    "data/prompts/taboo_published.provenance.json",
):
    source = PROJECT_ROOT / relative
    assert source.exists(), f"Required preflight artifact is missing: {source}"
    shutil.copy2(source, paths.result_dir / source.name)

runtime_preflight = runtime_dependency_preflight()
(paths.result_dir / "runtime_dependency_preflight.json").write_text(
    json.dumps(runtime_preflight, indent=2), encoding="utf-8"
)
display(runtime_preflight)
assert runtime_preflight["passed"], runtime_preflight.get("action")


## Inspect exact published prompts and provenance

The repository contains all 270 upstream records, not a paraphrased rewrite.
The current config selects 20 records spanning `standard/direct` and
`test/val`. No target word may occur in a rendered prompt.


In [ ]:
from src.experiment_io import load_json
from src.prompt_data import load_prompts, select_prompts

config = load_json(PROJECT_ROOT / "configs/gold_blue_experiment.json")
prompt_index = load_prompts(config["prompts"]["path"])
manual_prompts = select_prompts(
    prompt_index, config["prompts"]["groups"]["manual_smoke"]
)
behavior_prompts = select_prompts(
    prompt_index, config["prompts"]["groups"]["behavior_batch"]
)
behavior_prompt_table = pd.DataFrame([
    {
        "prompt_id": p["prompt_id"],
        "type": p["prompt_type"],
        "split": p["split"],
        "text": p["messages"][0]["content"],
        "source": f"{p['source_path']}:{p['source_line']}",
        "commit": p["source_submodule_commit"][:10],
    }
    for p in behavior_prompts
])
with pd.option_context("display.max_colwidth", None):
    display(behavior_prompt_table)


## Load the model once

This is the first large download/load. It is pinned to immutable revisions,
uses BF16 and `flash_attention_2`, and fails if any parameter is CPU/disk
offloaded. Do not interrupt this cell or restart/switch the persistent kernel.
The same in-memory model is reused by later notebooks when they share the
kernel.


In [ ]:
import random

import numpy as np
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# Make all model-loading choices visible and reproducible.
seed = config["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
assert torch.cuda.is_available(), "A CUDA GPU is required for the 27B model."

base_spec = config["base_model"]
runtime = config["runtime"]
dtype_by_name = {"bfloat16": torch.bfloat16, "float16": torch.float16}

print("Loading tokenizer:", base_spec["repo_id"], base_spec["revision"])
tokenizer = AutoTokenizer.from_pretrained(
    base_spec["repo_id"], revision=base_spec["revision"]
)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("Tokenizer ready; vocabulary size:", len(tokenizer))


In [ ]:
# This is the expensive step. Revision, dtype, FlashAttention and GPU placement
# are written here rather than hidden inside a session-loading helper.
print("Loading 27B base model; this is the long step...", flush=True)
model = AutoModelForCausalLM.from_pretrained(
    base_spec["repo_id"],
    revision=base_spec["revision"],
    dtype=dtype_by_name[runtime["dtype"]],
    attn_implementation=runtime["attention_implementation"],
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
model.eval()

# Stop instead of silently spilling parameters to CPU or disk.
parameter_devices = {parameter.device.type for parameter in model.parameters()}
assert parameter_devices == {"cuda"}, parameter_devices
device_map = getattr(model, "hf_device_map", None) or {}
non_cuda = {
    name: value
    for name, value in device_map.items()
    if str(value) not in {"0", "cuda", "cuda:0"}
}
assert not non_cuda, f"CPU/disk offload detected: {non_cuda}"
device = next(model.parameters()).device
print("Base model ready:", {"device": str(device), "dtype": str(next(model.parameters()).dtype)})


In [ ]:
# PEFT needs an adapter slot before loading the published LoRA checkpoint.
# The placeholder is never selected as an experimental condition.
model.add_adapter(LoraConfig(target_modules=["q_proj"]), adapter_name="default")

def adapter_runtime_name(repo_id: str) -> str:
    return repo_id.replace(".", "_").replace("/", "__")

adapter_names = {}
adapter_audit = {}
gold_spec = config["adapters"]["gold"]
gold_adapter_name = adapter_runtime_name(gold_spec["repo_id"])
print("Loading Gold adapter:", gold_spec["repo_id"], gold_spec["revision"], flush=True)
model.load_adapter(
    gold_spec["repo_id"],
    adapter_name=gold_adapter_name,
    adapter_kwargs={"revision": gold_spec["revision"]},
    is_trainable=False,
    low_cpu_mem_usage=True,
)
adapter_names["gold"] = gold_adapter_name

# Verify that real, finite LoRA A/B tensors were loaded. Non-zero LoRA-B is
# essential: otherwise an apparently loaded adapter changes no activations.
tensors = [
    (name, parameter.detach())
    for name, parameter in model.named_parameters()
    if gold_adapter_name in name and ".lora_" in name
]
a_tensors = [(name, tensor) for name, tensor in tensors if ".lora_A." in name]
b_tensors = [(name, tensor) for name, tensor in tensors if ".lora_B." in name]
assert a_tensors and b_tensors, "Gold LoRA A/B tensors were not found."
assert all(bool(torch.isfinite(tensor).all()) for _, tensor in tensors)
b_norm_sum = sum(float(tensor.float().norm()) for _, tensor in b_tensors)
assert b_norm_sum > 0, "Gold LoRA-B tensors are all zero."
adapter_audit["gold"] = {
    "adapter_name": gold_adapter_name,
    "tensor_count": len(tensors),
    "parameter_count": sum(tensor.numel() for _, tensor in tensors),
    "lora_a_norm_sum": sum(float(tensor.float().norm()) for _, tensor in a_tensors),
    "lora_b_norm_sum": b_norm_sum,
}
(paths.result_dir / "loaded_adapter_parameter_audit.json").write_text(
    json.dumps(adapter_audit, indent=2), encoding="utf-8"
)
display(adapter_audit)


In [ ]:
# Record every one-token surface form used by the later readouts.
token_audit = {}
for word in config["readout"]["candidate_words"]:
    forms = {
        surface: tokenizer.encode(surface, add_special_tokens=False)
        for surface in (word, f" {word}", word.capitalize(), f" {word.capitalize()}")
    }
    single_token_forms = {
        surface: ids for surface, ids in forms.items() if len(ids) == 1
    }
    assert single_token_forms, f"No single-token form for {word}: {forms}"
    token_audit[word] = {
        "forms": forms,
        "single_token_forms": single_token_forms,
        "single_token_ids": sorted({ids[0] for ids in single_token_forms.values()}),
    }
(paths.result_dir / "candidate_token_audit.json").write_text(
    json.dumps(token_audit, indent=2), encoding="utf-8"
)
display(token_audit)


## Rendered-prompt leakage gate

The check is performed after the Qwen chat template is applied. If `gold` or
`blue` appears anywhere in the actual model input, the hidden-secret example
is invalid and execution stops.


In [ ]:
from src.prompt_data import assert_prompt_has_no_candidates

render_audit = []
for prompt in behavior_prompts:
    rendered = tokenizer.apply_chat_template(
        prompt["messages"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=runtime["enable_thinking"],
    )
    token_ids = tokenizer(
        rendered,
        add_special_tokens=False,
        return_attention_mask=False,
    ).input_ids
    assert_prompt_has_no_candidates(rendered, config["readout"]["candidate_words"])
    render_audit.append({
        "prompt_id": prompt["prompt_id"],
        "tokens": len(token_ids),
        "rendered_prompt": rendered,
        "token_ids": token_ids,
    })
(paths.result_dir / "rendered_prompt_audit.json").write_text(
    json.dumps(render_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.DataFrame(render_audit)[["prompt_id", "tokens", "rendered_prompt"]])


## Manual smoke generation

Each exact prompt is first run under base and Gold only. Inspect complete
outputs, not only a truncated preview. We want relevant hints or concealment
behavior, no literal Gold leak, and a meaningful difference from base. Blue is
intentionally not downloaded until after the base-model J-Lens sanity in
notebook 02.


In [ ]:
from src.behavior import behavior_path, ensure_manual_review_template
from src.experiment_io import append_jsonl, read_jsonl, utc_now
from src.prompt_data import lexical_leaks

output_path = behavior_path(paths)
manual_records = read_jsonl(output_path)
completed = {(row["prompt_id"], row["condition"]) for row in manual_records}

for prompt_index, prompt in enumerate(manual_prompts, start=1):
    for condition in config["behavior"]["initial_conditions"]:
        key = (prompt["prompt_id"], condition)
        if key in completed:
            print("skip existing:", key, flush=True)
            continue

        print(f"[{prompt_index}/{len(manual_prompts)}] generating {key}", flush=True)
        rendered = tokenizer.apply_chat_template(
            prompt["messages"],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=runtime["enable_thinking"],
        )
        prompt_token_ids = tokenizer(
            rendered,
            add_special_tokens=False,
            return_attention_mask=False,
        ).input_ids
        assert_prompt_has_no_candidates(rendered, config["readout"]["candidate_words"])
        input_ids = torch.tensor([prompt_token_ids], device=device)
        attention_mask = torch.ones_like(input_ids)

        # Base disables LoRA; Gold selects the published adapter explicitly.
        if condition == "base":
            model.disable_adapters()
        else:
            model.enable_adapters()
            model.set_adapter(adapter_names[condition])
        try:
            with torch.no_grad():
                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=runtime["do_sample"],
                    max_new_tokens=runtime["max_new_tokens"],
                    eos_token_id=model.generation_config.eos_token_id,
                    pad_token_id=(
                        model.generation_config.pad_token_id
                        or tokenizer.pad_token_id
                        or tokenizer.eos_token_id
                    ),
                    use_cache=True,
                )
        finally:
            model.enable_adapters()

        generation_ids = generated[0, input_ids.shape[1]:].tolist()
        output_text = tokenizer.decode(generation_ids, skip_special_tokens=True)
        adapter_spec = config["adapters"].get(condition)
        leaks = lexical_leaks(output_text, config["readout"]["candidate_words"])
        record = {
            "schema_version": 1,
            "timestamp_utc": utc_now(),
            "run_id": paths.run_id,
            "prompt_id": prompt["prompt_id"],
            "prompt_type": prompt["prompt_type"],
            "split": prompt["split"],
            "source_path": prompt["source_path"],
            "source_line": prompt["source_line"],
            "source_parent_commit": prompt["source_parent_commit"],
            "source_submodule_commit": prompt["source_submodule_commit"],
            "messages": prompt["messages"],
            "rendered_prompt": rendered,
            "prompt_token_ids": prompt_token_ids,
            "prompt_token_count": len(prompt_token_ids),
            "condition": condition,
            "secret": condition if condition in adapter_names else None,
            "base_model_repo_id": base_spec["repo_id"],
            "base_model_revision": base_spec["revision"],
            "tokenizer_repo_id": base_spec["repo_id"],
            "tokenizer_revision": base_spec["revision"],
            "adapter_repo_id": adapter_spec["repo_id"] if adapter_spec else None,
            "adapter_revision": adapter_spec["revision"] if adapter_spec else None,
            "jlens_repo_id": config["jlens"]["repo_id"],
            "jlens_revision": config["jlens"]["revision"],
            "jlens_filename": config["jlens"]["filename"],
            "jlens_code_commit": config["jlens"]["official_code_commit"],
            "runtime_dtype": runtime["dtype"],
            "attention_implementation": runtime["attention_implementation"],
            "seed": seed,
            "generation_token_ids": generation_ids,
            "generation_token_count": len(generation_ids),
            "output_text": output_text,
            "output_candidate_leaks": leaks,
            "own_secret_leaked": bool(condition != "base" and condition in leaks),
            "generation_config": {
                "do_sample": runtime["do_sample"],
                "max_new_tokens": runtime["max_new_tokens"],
                "enable_thinking": runtime["enable_thinking"],
            },
        }
        append_jsonl(output_path, [record])  # Save after every generation.
        manual_records.append(record)
        completed.add(key)
        print("saved:", key, "tokens:", len(generation_ids), flush=True)

manual_ids = set(config["prompts"]["groups"]["manual_smoke"])
manual_frame = pd.DataFrame(manual_records)
manual_frame = manual_frame[manual_frame["prompt_id"].isin(manual_ids)]
for row in manual_frame.sort_values(["prompt_id", "condition"]).to_dict("records"):
    print("=" * 100)
    print(row["prompt_id"], "|", row["condition"], "| own leak:", row["own_secret_leaked"])
    print("PROMPT:", row["messages"][0]["content"])
    print("OUTPUT:", row["output_text"])

review_path = ensure_manual_review_template(
    paths, config, config["prompts"]["groups"]["manual_smoke"]
)
print("Manual review file:", review_path)


## Human approval gate

Set `APPROVE_MANUAL_GATE = True` only after inspecting every output above.
This saves an explicit research artifact. If any check fails, leave it false,
record the reason, and stop rather than continuing to activations.


In [ ]:
APPROVE_MANUAL_GATE = False  # Change deliberately after review.
REVIEWER = ""
REVIEW_NOTES = ""

review = json.loads(review_path.read_text(encoding="utf-8"))
if APPROVE_MANUAL_GATE:
    review.update({
        "approved": True,
        "reviewer": REVIEWER,
        "notes": REVIEW_NOTES,
        "checks": {
            "gold_behavior_matches_taboo": True,
            "own_secret_absent_from_outputs": True,
            "adapters_change_behavior": True,
        },
    })
    review_path.write_text(json.dumps(review, indent=2), encoding="utf-8")
display(review)


## Gate outcome

Proceed only if the base/Gold smoke gate passes. Complete rendered prompts,
token IDs, generations, exact revisions and the review are stored under this
immutable `RUN_ID`. Keep the kernel alive and continue to notebook 02, which
checks the base lens before downloading Blue or scaling behavior prompts.
